# MDM Classic vs Published Signals -- Interactive Explorer

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
%matplotlib inline

from core.data_loader import DataLoader
from core.signal_loader import load_signal_fixture
from core.signal_comparator import (
    extract_model_signals, align_signals, compare_signals,
    classify_divergences, STATE_TO_SIGNAL
)
from strategies.mdm_classic import MDMEngine
from strategies.mdm_classic.config import MDMConfig

# Configuration -- change these to zoom into specific periods
start_date = '2019-01-01'
end_date = '2026-12-31'

In [ ]:
nasdaq_df = DataLoader('nasdaq').load(start_date='2017-01-01')
published = load_signal_fixture('data/signals/nasdaq_signals.csv')
engine = MDMEngine()
results = engine.run(nasdaq_df)
model_signals = extract_model_signals(results)
print(f"NASDAQ data: {len(nasdaq_df)} rows, {nasdaq_df['date'].min()} to {nasdaq_df['date'].max()}")
print(f"Published signals: {len(published)} rows")
print(f"Model signals: {len(model_signals)} transitions")

In [ ]:
metrics = compare_signals(model_signals, published)
aligned = align_signals(model_signals, published)
classified = classify_divergences(aligned, model_signals, results, MDMConfig())
print(f"Match rate: {metrics['match_rate']:.1f}%")
print(f"Total published: {metrics['total_published']}, Matched: {metrics['total_matched']}")
for sig_type, info in metrics['per_type'].items():
    print(f"  {sig_type}: {info['matched']}/{info['published']} ({info['rate']:.1f}%)")

## Summary Statistics

In [ ]:
divergences = classified[classified['divergence_type'].notna()]
if len(divergences) > 0:
    print(f"Total divergences: {len(divergences)}")
    print(divergences['divergence_type'].value_counts().to_string())
else:
    print("No divergences found (perfect match)")

## Full Signal Overlay Chart

In [ ]:
SIGNAL_COLORS = {
    'Buy': '#2ca02c',
    'Sell': '#d62728',
    'Cash': '#7f7f7f',
}

def expand_to_daily(signals_df, date_range):
    daily = pd.DataFrame({'date': date_range})
    daily = daily.merge(signals_df[['date', 'signal']], on='date', how='left')
    daily['signal'] = daily['signal'].ffill()
    return daily

def find_divergence_periods(pub_signals, mod_signals):
    divergent = pub_signals != mod_signals
    groups = divergent.ne(divergent.shift()).cumsum()
    periods = []
    for _, group in divergent.groupby(groups):
        if group.iloc[0]:
            periods.append((group.index[0], group.index[-1]))
    return periods

def render_signal_track(ax, daily_df, label):
    dates = daily_df['date'].values
    signals = daily_df['signal'].values
    if len(signals) == 0:
        return
    current_signal = signals[0]
    start_idx = 0
    for i in range(1, len(signals)):
        if signals[i] != current_signal or i == len(signals) - 1:
            end_idx = i if signals[i] != current_signal else i + 1
            if pd.notna(current_signal) and current_signal in SIGNAL_COLORS:
                s = pd.Timestamp(dates[start_idx])
                e = pd.Timestamp(dates[min(end_idx, len(dates) - 1)])
                ax.axvspan(s, e, alpha=0.7, color=SIGNAL_COLORS[current_signal], linewidth=0)
            current_signal = signals[i]
            start_idx = i
    if pd.notna(current_signal) and current_signal in SIGNAL_COLORS:
        s = pd.Timestamp(dates[start_idx])
        e = pd.Timestamp(dates[-1])
        ax.axvspan(s, e, alpha=0.7, color=SIGNAL_COLORS[current_signal], linewidth=0)
    ax.set_ylabel(label, fontsize=10, fontweight=600)
    ax.set_yticks([])

# Filter to date range
comp_df = results[(results['date'] >= start_date) & (results['date'] <= end_date)].copy()
date_range = comp_df['date'].values
daily_pub = expand_to_daily(published, date_range)
daily_mod = expand_to_daily(model_signals, date_range)

fig = plt.figure(figsize=(18, 9))
fig.patch.set_facecolor('#ffffff')
gs = GridSpec(3, 1, height_ratios=[4, 1, 1], hspace=0.05)

ax_price = fig.add_subplot(gs[0])
ax_price.plot(comp_df['date'], comp_df['close'], color='#1f1f1f', linewidth=0.8)
ax_price.set_ylabel('NASDAQ', fontsize=10, fontweight=600)
ax_price.grid(True, color='#e0e0e0', alpha=0.5)
ax_price.set_facecolor('#ffffff')

pub_s = daily_pub.set_index('date')['signal']
mod_s = daily_mod.set_index('date')['signal']
common_idx = pub_s.index.intersection(mod_s.index)
periods = find_divergence_periods(pub_s.reindex(common_idx), mod_s.reindex(common_idx))
for s, e in periods:
    ax_price.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.12, color='#d62728', linewidth=0)

for sp in ['top', 'right']:
    ax_price.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax_price.spines[sp].set_color('#cccccc')

ax_pub = fig.add_subplot(gs[1], sharex=ax_price)
render_signal_track(ax_pub, daily_pub, 'Published')
ax_pub.set_facecolor('#ffffff')
for sp in ['top', 'right']:
    ax_pub.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax_pub.spines[sp].set_color('#cccccc')

ax_mod = fig.add_subplot(gs[2], sharex=ax_price)
render_signal_track(ax_mod, daily_mod, 'Model (Classic)')
ax_mod.set_facecolor('#ffffff')
for sp in ['top', 'right']:
    ax_mod.spines[sp].set_visible(False)
for sp in ['left', 'bottom']:
    ax_mod.spines[sp].set_color('#cccccc')

plt.setp(ax_price.get_xticklabels(), visible=False)
plt.setp(ax_pub.get_xticklabels(), visible=False)
locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
ax_mod.xaxis.set_major_locator(locator)
ax_mod.xaxis.set_major_formatter(formatter)

sy = pd.Timestamp(date_range[0]).year
ey = pd.Timestamp(date_range[-1]).year
fig.suptitle(f'MDM Classic vs Published Signals -- NASDAQ {sy}-{ey}', fontsize=14, fontweight=600, y=0.98)

plt.tight_layout()
plt.show()

## Zoom: Specific Divergence Periods

In [ ]:
filter_type = None  # Change to 'TIMING', 'STRUCTURAL', 'THRESHOLD', or 'IRREPRODUCIBLE'
if filter_type:
    display(divergences[divergences['divergence_type'] == filter_type])
else:
    display(divergences)